# Fruits and Vegetables Freshness Classification

This notebook version uses the same reusable code in `src/` to train and evaluate a TensorFlow/Keras CNN for all 20 fresh/rotten classes.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)

## 1. Discover Images and Create Splits

The dataset is nested as `dataset/Fruits/<class>` and `dataset/Vegetables/<class>`. Each class folder becomes one model output class.

In [ ]:
from src.config import *
from src.data import discover_images, make_class_indices, add_label_indices, stratified_train_val_test_split, save_class_indices, save_splits

manifest = discover_images(DATASET_DIR)
class_indices = make_class_indices(manifest["label"])
manifest = add_label_indices(manifest, class_indices)

train_df, val_df, test_df = stratified_train_val_test_split(
    manifest,
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    random_seed=RANDOM_SEED,
)

save_class_indices(class_indices, CLASS_INDICES_PATH)
save_splits(train_df, val_df, test_df, (TRAIN_SPLIT_PATH, VAL_SPLIT_PATH, TEST_SPLIT_PATH))

print(f"Images: {len(manifest):,}")
print(f"Classes: {len(class_indices)}")
manifest["label"].value_counts().sort_index()

## 2. Visualize the Dataset

In [ ]:
from src.visualize import plot_class_distribution, plot_sample_images

plot_class_distribution(manifest, PLOTS_DIR / "class_distribution.png")
plot_sample_images(manifest, IMAGE_SIZE, PLOTS_DIR / "sample_images.png")

print(PLOTS_DIR / "class_distribution.png")
print(PLOTS_DIR / "sample_images.png")

## 3. Build the CNN

The CNN uses convolution layers for feature extraction, pooling for spatial reduction, dense layers for classification, ReLU activations for non-linearity, and softmax for class probabilities.

In [ ]:
import tensorflow as tf
from src.model import build_cnn_model, print_parameter_report

tf.keras.utils.set_random_seed(RANDOM_SEED)
model = build_cnn_model(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    num_classes=len(class_indices),
    learning_rate=LEARNING_RATE,
)
model.summary()
print_parameter_report(model)

## 4. Train the Model

In [ ]:
from src.data import ImageClassificationSequence

train_sequence = ImageClassificationSequence(train_df, class_indices, IMAGE_SIZE, BATCH_SIZE, shuffle=True)
val_sequence = ImageClassificationSequence(val_df, class_indices, IMAGE_SIZE, BATCH_SIZE, shuffle=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
]

history = model.fit(
    train_sequence,
    validation_data=val_sequence,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## 5. Save Model and Training Curves

In [ ]:
import pandas as pd
from src.visualize import plot_training_curves

model.save(FINAL_MODEL_PATH)
pd.DataFrame(history.history).to_csv(PLOTS_DIR / "training_history.csv", index=False)
plot_training_curves(history, PLOTS_DIR / "training_curves.png")

print(FINAL_MODEL_PATH)
print(PLOTS_DIR / "training_curves.png")

## 6. Evaluate on the Test Set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from src.visualize import plot_confusion_matrix_heatmap

test_sequence = ImageClassificationSequence(test_df, class_indices, IMAGE_SIZE, BATCH_SIZE, shuffle=False)
test_loss, test_accuracy = model.evaluate(test_sequence, verbose=1)
probabilities = model.predict(test_sequence, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
y_true = test_df["label_index"].to_numpy()

index_to_class = {index: class_name for class_name, index in class_indices.items()}
class_names = [index_to_class[index] for index in range(len(index_to_class))]

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix_heatmap(cm, class_names, PLOTS_DIR / "confusion_matrix_heatmap.png")

## 7. Single Image Prediction

In [ ]:
from src.predict import predict_single_image

# Replace this with any image path from the dataset or your own test image.
example_image = manifest.iloc[0]["file_path"]
predicted_class, confidence = predict_single_image(example_image, FINAL_MODEL_PATH, CLASS_INDICES_PATH, IMAGE_SIZE)

print(f"Image: {example_image}")
print(f"Predicted class: {predicted_class}")
print(f"Prediction confidence: {confidence:.2%}")